In [1]:
# !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt -O input.txt

In [2]:
with open('input.txt', 'r') as f:
    text = f.read()

In [ ]:
# Hyper Params

block_size = 256 # same as context length
embedding_dim = 384 
head_size = 64
num_heads = 6

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

/home/arunkant/miniconda3/envs/nanogpt/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for i,c in enumerate(chars)}
def encode(s):
    return [stoi[c] for c in s]
def decode(s):
    return ''.join([itos[i] for i in s])
data = torch.tensor(encode(text), dtype=torch.long)
ninghty_percent = int(0.9*len(data))
train_data, val_data = data[:ninghty_percent], data[ninghty_percent:]

In [1]:
def get_batch(split, batch_size=4):
    data = train_data if split == 'train' else val_data
    indeces = torch.randint(0, len(data)-block_size, (batch_size, ))
    X = torch.stack([data[i:i+block_size] for i in indeces])
    Y = torch.stack([data[i+1:i+block_size+1] for i in indeces])
    return X, Y
def sample_with_prompt(model, prompt, max_new_tokens):
    return decode([ x.item() for x in model.generate(torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0), max_new_tokens, block_size)[0]])

In [7]:
class Head(nn.Module):
    def __init__(self, embedding_dim, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)
        kt = k.transpose(-2, -1) # dim1 = T, dim2 = head_size
        wei = q@kt # (B, T, T)
        wei = wei * (C ** -0.5) # <-- Add this line for stability!
        
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        return wei @ v # (B, T, head_size)
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, head_size, num_heads, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(embedding_dim, head_size, block_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, embedding_dim)

    def forward(self, x):
        outputs = torch.cat([head(x) for head in self.heads], dim=-1) # B, T, num_heads * head_size
        return self.proj(outputs)

class Block(nn.Module):
    def __init__(self, embedding_dim, head_size, num_heads, block_size):
        super().__init__()
        self.mha = MultiHeadAttention(embedding_dim, head_size, num_heads, block_size)
        self.layer1 = nn.Linear(embedding_dim, 4 * embedding_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(4 * embedding_dim, embedding_dim)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

    def feed_forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x
    
    def forward(self, x):
        x = x + self.mha(self.norm1(x)) # skip connections
        x = x + self.feed_forward(self.norm2(x))
        return x

class ToyGPT(nn.Module):
    def __init__(self, block_size, vocab_size, embedding_dim, head_size, num_heads):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size, embedding_dim)
        self.pos_embedding_table = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(*[Block(embedding_dim, head_size, num_heads, block_size) for _ in range(4)])
        self.lm_head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        tok_emb = self.embedding_table(x)
        pos = torch.arange(T, device=x.device)
        pos_embedding = self.pos_embedding_table(pos)
        x = tok_emb + pos_embedding
        x = self.blocks(x)
        logits = self.lm_head(x) # B, T, vocab_size
        B, T, C = logits.shape

        if targets is None:
            loss = None
        else:
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, block_size, temperature=1.0, top_k=None):
        device = next(self.parameters()).device
        idx = idx.to(device)
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # B, T
            logits, _ = self(idx_cond) # B, T, C
            logits = logits[:, -1, :] # B, C

            # Apply temperature
            logits = logits / temperature

            if top_k is not None:
                # Find the kth highest logit value for each sequence in the batch
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                
                # Mask out everything below that kth value with negative infinity
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1) # B, 1
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    

In [8]:
model = ToyGPT(block_size, vocab_size, embedding_dim, head_size, num_heads)
model = model.to(device)
sample_with_prompt(model, '\n', 128)

"\n TLbVDkoBs!CCxheQaSERriUh-IXHDjQhb;qyhfqOFnHVNN,-D&ymLwo\n3Uyx:LvP xIDCzqN,jzwXsADaxegV'zScSvedTFFzgY'WUztEXHXjwTWEehmuz;npWnSLx$"

In [ ]:
# Training
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32
max_iters = 1000


In [ ]:
print(f"Training on: {device}")
for iter in range(max_iters):
    # 1. Grab a batch of data
    xb, yb = get_batch('train', batch_size) # xb and yb have shape (batch_size, block_size)
    xb = xb.to(device)
    yb = yb.to(device)
    
    # 2. Evaluate the loss
    logits, loss = model(xb, yb)
    
    # 3. Clear the old gradients
    optimizer.zero_grad(set_to_none=True)
    
    # 4. Calculate the new gradients (backpropagation)
    loss.backward()
    
    # 5. Update the weights
    optimizer.step()
    
    if iter % 100 == 0:
        print(f"Step {iter}: Loss {loss.item():.4f}")

Training on: cuda
Step 0: Loss 1.6150
Step 100: Loss 1.5678
Step 200: Loss 1.6828
Step 300: Loss 1.5666
Step 400: Loss 1.6022
Step 500: Loss 1.5365
Step 600: Loss 1.5217
Step 700: Loss 1.3181
Step 800: Loss 1.6143
Step 900: Loss 1.4741


In [19]:
print(sample_with_prompt(model, '\n', 512))


Both do it sbane hate word and iund.

CAPULET:
But to your this life; and yet no show;
Cupident, we makes them sing to me
Behold, he's hear his high o' But, being

ESCALUS:
Here ever faint sperfect sisten
Before lies thee proice, my is solemn age,
Yes farm thing of me,
New take friends my block in a briber
Advifn, thou may not a grief;
And our hand is Contence conjed forth,
Their. A sir, whick have swear,
And with me petor with ingolain,
And grow did confintening: them your honour fathers
No do bay to cheat


In [16]:
total_params = sum(p.numel() for p in model.parameters())
total_params

7241537

In [13]:
print(sample_with_prompt(model, '\n', 128))


Cried, be shal ark becoves;
Are I men exe a gave and that tay heavainy
My ariie. foremor Voul so,
I exetized; beal gook of a gla
